<a href="https://colab.research.google.com/github/BelenUrdangarin/Integrador-Urdangarin---Olivera/blob/main/Olivera_Urdangarin_Integrador_(Pre_entrega_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Trabajo sobre un modelo de aprendizaje supervisado.- Ajuste de modelos de clasificación o regresión.- Evaluación de los modelos.- Optimización de hiperparámetros.

##Energia renovable

###Objetivo:

El objetivo del proyecto es analizar y predecir cuál es la fuente de energía renovable más adecuada para cada región y continente, en función de variables climáticas, demográficas y económicas, con el fin de orientar estrategias sostenibles de desarrollo energético a nivel regional

Objetivo del modelo - DF de datos climáticos
Clasificar a los países en tres niveles de impacto climático:

Bajo

Medio

Alto

Esto se basará en variables climáticas y de calidad del aire.

- Del dataset de prod ren trataremos de estimar el progreso en el uso de energias renovables y como cada pais o region seguira por ese camino.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [5]:
clim_pais_fil = pd.read_csv("https://raw.githubusercontent.com/BelenUrdangarin/Integrador-Urdangarin---Olivera/refs/heads/main/clim_pais_filtrado.csv", sep=',',on_bad_lines='skip',na_values="..") #Importo los datos
prod_ren_fil = pd.read_csv("https://raw.githubusercontent.com/BelenUrdangarin/Integrador-Urdangarin---Olivera/refs/heads/main/prod_ren_fil.csv", sep=',',on_bad_lines='skip',na_values="..") #Importo los datos


In [3]:
print(clim_pais_fil.columns)  # Muestra las columnas disponibles


Index(['country', 'latitude', 'longitude', 'last_updated',
       'temperature_celsius', 'condition_text', 'wind_mph', 'pressure_mb',
       'precip_mm', 'cloud', 'uv_index', 'air_quality_Carbon_Monoxide',
       'air_quality_PM2.5', 'air_quality_PM10', 'continent'],
      dtype='object')


La variables seleccionadas (x) serán todas aquellas que podemos utilizar para determinar como se encuentra la situacion climatica de un país:
- Temperatura
- Indice de rayos uv
- Niveles de CO2 y Monoxido de Carbono

La variable objetivo (y) se llamará *Nivel de impacto* y se creará para dividir los datos en tres clases segun el nivel *bajo*, *medio* o *alto* de impacto climatico. Para esto utilizaremos qcut() que nos permite dividir cada grupo en un mismo numero aproximado de elementos y los transforma en una variable categorica.

In [7]:
#Vamos a crear el indice de impacto a partir de la medio normalizada de los factores
scal = MinMaxScaler()
cols = ['temperature_celsius', 'uv_index', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_Carbon_Monoxide']

clim_norm = scal.fit_transform(clim_pais_fil[cols])
clim_pais_fil['Impacto'] = clim_norm.mean(axis=1)

# Creamos las categorías de de la variable objetivo
clim_pais_fil['Impacto_cat'] = pd.qcut(clim_pais_fil['Impacto'], 3, labels=['Bajo', 'Medio', 'Alto'])

Lo que hicimos fue:
1. Crear la columna *Impacto* a partir de la media normalizada de las categorias seleccionadas para ver cual es el pedo de cada pais en cada uno de estos factores.

2. Dividir la columna de *Impacto* en tres grupos con aproximadamente la misma cantidad de paises.

3. Le asignamos etiquetas de *bajo*, *medio* y *alto*

4. Finalmente, creamos una nueva columa llamada *Impacto_cat* que le da a cada pais su nivel de impacto.

In [10]:
X = clim_pais_fil[cols] #Determinamos las variables seleccionadas
y = clim_pais_fil['Impacto_cat'] #Y las objetivo

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Dividimos los daos en un 80% para entrenar y 20% para evaluar el desempeño
#Definimos el random state para que la sivision siempre sea la misma
#Creamos el modelo de Random forest y lo entrenamos con los datos de entrenamiento
#Para que aprenda
modelo = RandomForestClassifier(random_state=42)
modelo.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

Elegimos *Random Forest* porque es robusto frente a outliers y ruido, no necesita normalización y es el mejor para evitar el overfitting.

In [11]:
y_pred = modelo.predict(X_test)

print("Precisión:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Precisión: 0.9808873044167162
[[3320    0   48]
 [   0 3350   63]
 [  48   34 3235]]
              precision    recall  f1-score   support

        Alto       0.99      0.99      0.99      3368
        Bajo       0.99      0.98      0.99      3413
       Medio       0.97      0.98      0.97      3317

    accuracy                           0.98     10098
   macro avg       0.98      0.98      0.98     10098
weighted avg       0.98      0.98      0.98     10098

